# Deep Past Challenge — Inference v2 (Clean)

**Multi-model ensemble + Geo-Mean MBR + V3 Preprocessing**

### Features
1. **V3-aligned preprocessing**: ḫ→h, all gaps→`<gap>`, no `<big_gap>`
2. **Geo-mean MBR utility**: Optimizes `√(BLEU × chrF++)` — the actual competition metric
3. **Multi-model ensemble**: Pool candidates from ALL attached models
4. **Massive candidate pool**: 8 beam + 16 diverse samples = 24 per model
5. **Fixed postprocessing**: Fractions BEFORE forbidden char removal, gap unification
6. **Dual-GPU support**: Load models alternately on GPU 0/1 for memory efficiency
7. **FP32 only**: ByT5 float16 drops score 35→13, so we stay FP32

### Attach Your Models
- **Primary**: Your fine-tuned model (from training notebook output)
- **Secondary**: `mattiaangeli/byt5-akkadian-mbr-v2` (baseline 35.5)
- **Tertiary**: Any additional models for ensemble diversity

### Runtime
- GPU T4×2 | Internet OFF | Runtime limit 9 hours
- Expected: ~60-90 min for 4000 sentences with 2 models

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['TOKENIZERS_PARALLELISM'] = 'true'

import re, json, time, math, gc, warnings, logging, unicodedata
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from contextlib import nullcontext

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoConfig
from tqdm.auto import tqdm
import sacrebleu

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  GPU {i}: {name} ({mem:.1f} GB)")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# === Model Discovery ===
# Add paths for all models you want in the ensemble.
# The first model found becomes the "primary" model.
MODEL_SEARCH_PATHS = [
    # --- Fine-tuned Variant A (all data, broad) ---
    "/kaggle/input/akkadian-finetune-a/model/final",
    "/kaggle/input/akkadian-finetune-a/model/best",
    "/kaggle/input/models/manish756/akkadian-finetune-a/pytorch/default/1/model/final",
    "/kaggle/input/models/manish756/akkadian-models/pytorch/finetune_a/1/model/final",

    # --- Fine-tuned Variant B (high-quality, focused) ---
    "/kaggle/input/akkadian-finetune-b/model/final",
    "/kaggle/input/akkadian-finetune-b/model/best",
    "/kaggle/input/models/manish756/akkadian-finetune-b/pytorch/default/1/model/final",
    "/kaggle/input/models/manish756/akkadian-models/pytorch/finetune_b/1/model/final",

    # --- mattiaangeli baseline (35.5 score) ---
    "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1",
    "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr/pytorch/default/6",
    "/kaggle/input/byt5-akkadian-mbr-v2/pytorch/default/1",
    "/kaggle/input/byt5-akkadian-mbr-v2",

    # --- Previous models (for ensemble diversity) ---
    "/kaggle/input/models/manish756/akkadian-models/pytorch/default/2/model/final",
    "/kaggle/input/models/manish756/akkadian-models/pytorch/default/1/model/final",
    "/kaggle/input/models/manish756/akkadian-models/pytorch/phase_learning_from_scratch_byte_t5/1/model/best",
    "/kaggle/input/models/manwithacat/akkadian-byt5-synth-v1-0-0/pytorch/default/1",

    # --- Local testing ---
    "../models/byt5-akkadian/final",
    "../models/finetune_A/final",
    "../models/finetune_B/final",
]

# Discover available models
DISCOVERED_MODELS = []
_seen = set()
for p in MODEL_SEARCH_PATHS:
    rp = str(Path(p).resolve())
    if rp in _seen:
        continue
    if Path(p).exists():
        has_weights = (
            list(Path(p).glob('*.safetensors'))
            or list(Path(p).glob('*.bin'))
            or (Path(p) / 'config.json').exists()
        )
        if has_weights:
            DISCOVERED_MODELS.append(p)
            _seen.add(rp)

assert len(DISCOVERED_MODELS) > 0, (
    "No models found! Attach at least one model as a Kaggle dataset.\n"
    f"Searched: {MODEL_SEARCH_PATHS}"
)

# === Test data ===
TEST_PATH = "/kaggle/input/deep-past-initiative-machine-translation/test.csv"
if not Path(TEST_PATH).exists():
    TEST_PATH = "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv"
if not Path(TEST_PATH).exists():
    TEST_PATH = "../data/deep-past-initiative-machine-translation/test.csv"

OUTPUT_DIR = "/kaggle/working/" if Path("/kaggle").exists() else "./output/"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# === Generation parameters ===
MAX_SOURCE_LENGTH = 384
MAX_NEW_TOKENS = 256
NUM_BEAMS = 8
LENGTH_PENALTY = 1.3
REPETITION_PENALTY = 1.2

# === Batch size (auto-detect) ===
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram >= 30:
        BATCH_SIZE = 8
    elif vram >= 14:
        BATCH_SIZE = 2
    else:
        BATCH_SIZE = 1
else:
    BATCH_SIZE = 1

NUM_WORKERS = 2

# === MBR Decoding ===
USE_MBR = True
MBR_NUM_BEAM_CANDS = 8

# Diverse sampling: (temperature, top_p, num_samples)
MBR_SAMPLE_CONFIGS = [
    (0.5, 0.85, 4),
    (0.7, 0.90, 4),
    (1.0, 0.95, 4),
    (0.8, 0.80, 4),
]

MBR_UTILITY = 'geomean'  # 'geomean' or 'chrf'

PREFIX = "translate Akkadian to English: "

# === Summary ===
n_sample_cands = sum(c[2] for c in MBR_SAMPLE_CONFIGS)
n_per_model = MBR_NUM_BEAM_CANDS + n_sample_cands
n_total = n_per_model * len(DISCOVERED_MODELS)

print(f"\nDiscovered {len(DISCOVERED_MODELS)} model(s):")
for i, m in enumerate(DISCOVERED_MODELS):
    print(f"  [{i}] {m}")
print(f"\nMBR: {MBR_NUM_BEAM_CANDS} beam + {n_sample_cands} samples = {n_per_model}/model")
print(f"Total candidate pool: {n_per_model} x {len(DISCOVERED_MODELS)} = {n_total}")
print(f"Batch: {BATCH_SIZE} | Beams: {NUM_BEAMS} | Utility: {MBR_UTILITY}")
print(f"Test: {TEST_PATH}")

In [ ]:
# ============================================================
# V3 PREPROCESSING - Must match training preprocessing exactly
# ============================================================
# V3 rules from competition discussion:
#   1. All gaps unified to <gap> (no <big_gap>)
#   2. \u1e2b/\u1e2a -> h/H (test set uses h/H only)
#   3. Deduplicate sequential gaps
#   4. Minimal other changes (ByT5 handles raw bytes well)

_RE_BIG_GAP = re.compile(r'(\.{3,}|\u2026+)')
_RE_SMALL_GAP = re.compile(r'(xx+|\s+x\s+)')


def v3_preprocess(text):
    """V3-aligned preprocessing for inference."""
    if pd.isna(text) or not str(text).strip():
        return ''
    text = str(text)
    text = unicodedata.normalize('NFC', text)
    # \u1e2b/\u1e2a -> h/H
    text = text.replace('\u1e2b', 'h').replace('\u1e2a', 'H')
    # All gaps -> <gap>
    text = _RE_BIG_GAP.sub('<gap>', text)
    text = _RE_SMALL_GAP.sub('<gap>', text)
    text = text.replace('<big_gap>', '<gap>')
    # Deduplicate sequential gaps
    text = re.sub(r'(<gap>[\s\-]*){2,}', '<gap> ', text)
    # Clean whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def v3_preprocess_batch(texts):
    """Vectorized V3 preprocessing for a batch."""
    s = pd.Series(texts).fillna('').astype(str)
    s = s.apply(lambda t: unicodedata.normalize('NFC', t))
    s = s.str.replace('\u1e2b', 'h', regex=False)
    s = s.str.replace('\u1e2a', 'H', regex=False)
    s = s.str.replace(_RE_BIG_GAP, '<gap>', regex=True)
    s = s.str.replace(_RE_SMALL_GAP, '<gap>', regex=True)
    s = s.str.replace('<big_gap>', '<gap>', regex=False)
    s = s.str.replace(r'(<gap>[\s\-]*){2,}', '<gap> ', regex=True)
    s = s.str.replace(r'\s+', ' ', regex=True).str.strip()
    return s.tolist()


# Self-test
_test_inputs = [
    "um-ma ... a-na",
    "text <big_gap> more",
    "a-\u0161\u00f9r \u1e2ba-bu-\u1e2ba-lu",
    "<gap> <gap> text",
]
_test_out = v3_preprocess_batch(_test_inputs)
assert '<big_gap>' not in str(_test_out), f'<big_gap> leak: {_test_out}'
assert '\u1e2b' not in str(_test_out), f'\u1e2b leak: {_test_out}'
print("V3 Preprocessing OK:")
for inp, out in zip(_test_inputs, _test_out):
    print(f"  {inp} -> {out}")

In [ ]:
# ============================================================
# POSTPROCESSING - Fixed ordering (fractions BEFORE forbidden chars)
# ============================================================

_PP_PATTERNS = {
    'gap': re.compile(r'(\[x\]|\(x\)|\bx\b)', re.I),
    'big_gap': re.compile(r'(\.{3,}|\u2026|\[\.+\])'),
    'annotations': re.compile(r'\((fem|plur|pl|sing|singular|plural|\?|!)\.*\s*\w*\)', re.I),
    'repeated_words': re.compile(r'\b(\w+)(?:\s+\1\b)+'),
    'whitespace': re.compile(r'\s+'),
    'punct_space': re.compile(r'\s+([.,:])'),
    'repeated_punct': re.compile(r'([.,])\1+'),
}

_SUBSCRIPT_TRANS = str.maketrans(
    '\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089',
    '0123456789'
)
_SPECIAL_TRANS = str.maketrans('\u1e2b\u1e2a', 'hH')

# Forbidden chars - note / is here, so fractions must convert FIRST
_FORBIDDEN = '!?()"\u2014\u2014<>\u2308\u2309\u230a\u230b[]+\u02be/;'
_FORBIDDEN_TRANS = str.maketrans('', '', _FORBIDDEN)

# N-gram repetition patterns (4-gram down to 2-gram)
_NGRAM_PATS = []
for n in range(4, 1, -1):
    p = r'\b((?:\w+\s+){' + str(n - 1) + r'}\w+)(?:\s+\1\b)+'
    _NGRAM_PATS.append(re.compile(p))


def postprocess_batch(translations):
    """Postprocess model outputs. Order matters!"""
    s = pd.Series(translations).fillna('').astype(str)

    # Step 1: Basic character cleanup
    s = s.str.translate(_SPECIAL_TRANS)
    s = s.str.translate(_SUBSCRIPT_TRANS)
    s = s.str.replace(_PP_PATTERNS['whitespace'], ' ', regex=True).str.strip()

    # Step 2: Gap normalization (V3: all -> <gap>)
    s = s.str.replace(_PP_PATTERNS['gap'], '<gap>', regex=True)
    s = s.str.replace(_PP_PATTERNS['big_gap'], '<gap>', regex=True)
    s = s.str.replace('<big_gap>', '<gap>', regex=False)
    s = s.str.replace('<gap> <gap>', '<gap>', regex=False)

    # Step 3: Remove annotations
    s = s.str.replace(_PP_PATTERNS['annotations'], '', regex=True)

    # Step 4: Fractions -> Unicode (MUST come BEFORE forbidden char removal!)
    # Slash fractions
    s = s.str.replace(r'\b1/3\b', '\u2153', regex=True)   # 1/3
    s = s.str.replace(r'\b2/3\b', '\u2154', regex=True)   # 2/3
    s = s.str.replace(r'\b1/6\b', '\u2159', regex=True)   # 1/6
    s = s.str.replace(r'\b5/6\b', '\u215a', regex=True)   # 5/6
    s = s.str.replace(r'\b1/2\b', '\u00bd', regex=True)   # 1/2
    s = s.str.replace(r'\b1/4\b', '\u00bc', regex=True)   # 1/4
    s = s.str.replace(r'\b3/4\b', '\u00be', regex=True)   # 3/4
    # Decimal fractions
    s = s.str.replace(r'\b0\.5\b',    '\u00bd',          regex=True)
    s = s.str.replace(r'(\d+)\.5\b',  '\\g<1>\u00bd',  regex=True)
    s = s.str.replace(r'\b0\.25\b',   '\u00bc',          regex=True)
    s = s.str.replace(r'(\d+)\.25\b', '\\g<1>\u00bc',  regex=True)
    s = s.str.replace(r'\b0\.75\b',   '\u00be',          regex=True)
    s = s.str.replace(r'(\d+)\.75\b', '\\g<1>\u00be',  regex=True)

    # Step 5: Remove forbidden chars (protect gaps first)
    s = s.str.replace('<gap>', '\x00GAP\x00', regex=False)
    s = s.str.translate(_FORBIDDEN_TRANS)
    s = s.str.replace('\x00GAP\x00', ' <gap> ', regex=False)

    # Step 6: Remove repeated words and n-grams
    s = s.str.replace(_PP_PATTERNS['repeated_words'], r'\1', regex=True)
    for pat in _NGRAM_PATS:
        s = s.str.replace(pat, r'\1', regex=True)

    # Step 7: Fix punctuation and whitespace
    s = s.str.replace(_PP_PATTERNS['punct_space'], r'\1', regex=True)
    s = s.str.replace(_PP_PATTERNS['repeated_punct'], r'\1', regex=True)
    s = s.str.replace(_PP_PATTERNS['whitespace'], ' ', regex=True)
    s = s.str.strip().str.strip('-').str.strip()

    return s.tolist()


# === Self-test ===
_test_cases = [
    ("He paid 1.5 minas of silver.", "He paid 1\u00bd minas of silver."),
    ("0.5 shekel and 1/3 mina.",    "\u00bd shekel and \u2153 mina."),
    ("2/3 mina of gold.",           "\u2154 mina of gold."),
    ("<big_gap> text <big_gap>",    "<gap> text <gap>"),
]

_test_inputs = [t[0] for t in _test_cases]
_test_expected = [t[1] for t in _test_cases]
_test_results = postprocess_batch(_test_inputs)

all_ok = True
for inp, exp, got in zip(_test_inputs, _test_expected, _test_results):
    status = "\u2713" if got == exp else "\u2717"
    if got != exp:
        all_ok = False
    print(f"  {status} {inp} -> {got}")

assert all_ok, "Postprocessing self-test FAILED!"
print("\nPostprocessing self-test PASSED")

In [ ]:
# ============================================================
# MBR DECODING - Geo-Mean Utility (matches competition metric)
# ============================================================

_CHRFPP = sacrebleu.metrics.CHRF(word_order=2)
_BLEU = sacrebleu.metrics.BLEU(effective_order=True)


def _sim_chrf(a, b):
    """chrF++ similarity between two strings."""
    a, b = (a or '').strip(), (b or '').strip()
    if not a or not b:
        return 0.0
    return float(_CHRFPP.sentence_score(a, [b]).score)


def _sim_geomean(a, b):
    """Geometric mean of BLEU and chrF++ - matches competition metric."""
    a, b = (a or '').strip(), (b or '').strip()
    if not a or not b:
        return 0.0
    bleu = float(_BLEU.sentence_score(a, [b]).score)
    chrf = float(_CHRFPP.sentence_score(a, [b]).score)
    return math.sqrt(max(bleu, 0.1) * max(chrf, 0.1))


def mbr_pick(candidates, utility="geomean"):
    """Select best candidate via Minimum Bayes Risk decoding."""
    seen = set()
    unique = []
    for c in candidates:
        c = str(c).strip()
        if c and c not in seen:
            unique.append(c)
            seen.add(c)

    if len(unique) == 0:
        return ''
    if len(unique) == 1:
        return unique[0]

    sim_fn = _sim_geomean if utility == 'geomean' else _sim_chrf

    n = len(unique)
    scores = [0.0] * n
    for i in range(n):
        for j in range(n):
            if i != j:
                scores[i] += sim_fn(unique[i], unique[j])
        scores[i] /= max(1, n - 1)

    best_idx = max(range(n), key=lambda i: scores[i])
    return unique[best_idx]


# Self-test
_test_cands = [
    "Seal of Mannum-balum-A\u0161\u0161ur, son of \u1e62illi-Adad.",
    "Seal of Mannum-balum-A\u0161\u0161ur son of \u1e62ill-Adad.",
    "The seal of Mannum-balum-A\u0161\u0161ur, son of \u1e62illi-Adad.",
    "Mannum-balum-A\u0161\u0161ur son of \u1e62ill-Adad seal.",
]
_pick = mbr_pick(_test_cands, utility=MBR_UTILITY)
print(f"MBR defined ({MBR_UTILITY} utility). Self-test: {_pick}")

In [ ]:
# ============================================================
# LOAD MODELS - FP32 only (ByT5 float16 drops 35->13!)
# ============================================================

device_0 = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device_1 = torch.device('cuda:1' if torch.cuda.device_count() > 1 else device_0)

# Inspect all models
model_registry = []
for model_path in DISCOVERED_MODELS:
    try:
        cfg = AutoConfig.from_pretrained(model_path)
        model_registry.append({
            'path': model_path,
            'type': cfg.model_type,
            'name': '/'.join(Path(model_path).parts[-2:]),
            'prefix': PREFIX,
        })
        print(f"  \u2713 {model_registry[-1]['name']} (type={cfg.model_type})")
    except Exception as e:
        print(f"  \u2717 {model_path}: {e}")

assert len(model_registry) > 0, "No valid models found!"

# Load primary model onto GPU 0
print(f"\nLoading primary model: {model_registry[0]['path']}")
primary_tokenizer = AutoTokenizer.from_pretrained(model_registry[0]['path'])
primary_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_registry[0]['path'],
    torch_dtype=torch.float32,
)
primary_model = primary_model.to(device_0).eval()

# BetterTransformer for speed
try:
    from optimum.bettertransformer import BetterTransformer
    primary_model = BetterTransformer.transform(primary_model)
    print("  BetterTransformer: applied")
except Exception as e:
    print(f"  BetterTransformer: skipped ({e})")

try:
    torch.set_float32_matmul_precision('high')
except:
    pass

n_params = sum(p.numel() for p in primary_model.parameters())
dtype = next(primary_model.parameters()).dtype
print(f"  Loaded: {n_params/1e6:.0f}M params, dtype={dtype}, device={device_0}")
print(f"\nEnsemble size: {len(model_registry)} model(s)")
if len(model_registry) > 1:
    print("  (Additional models loaded on-demand during inference to save VRAM)")

In [ ]:
# ============================================================
# LOAD & PREPROCESS TEST DATA
# ============================================================

tokenizer = primary_tokenizer

test_df = pd.read_csv(TEST_PATH)
print(f"Test samples: {len(test_df)}")

# Apply V3 preprocessing
raw_texts = test_df['transliteration'].tolist()
processed = v3_preprocess_batch(raw_texts)
test_df['input_text'] = [PREFIX + t for t in processed]
test_df['token_len'] = test_df['input_text'].str.split().str.len()

print(f"\nInput length distribution:")
print(f"  Mean:   {test_df['token_len'].mean():.0f} words")
print(f"  Median: {test_df['token_len'].median():.0f} words")
print(f"  Max:    {test_df['token_len'].max()} words")

print(f"\nFirst 3 samples (after V3 preprocessing):")
for _, row in test_df.head(3).iterrows():
    print(f"  [{row['id']}] {row['input_text'][:100]}")

In [ ]:
# ============================================================
# DATASET + BUCKET-BATCH SAMPLER
# ============================================================

class AkkadianDataset(Dataset):
    def __init__(self, ids, texts):
        self.ids = ids
        self.texts = texts
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        return self.ids[i], self.texts[i]


class BucketBatchSampler(Sampler):
    """Groups similar-length inputs to minimize padding waste."""
    def __init__(self, dataset, batch_size, num_buckets=4):
        lengths = [len(t.split()) for _, t in dataset]
        sorted_indices = sorted(range(len(lengths)), key=lambda i: lengths[i])
        bucket_size = max(1, len(sorted_indices) // num_buckets)
        self.batches = []
        for b in range(num_buckets):
            start = b * bucket_size
            end = None if b == num_buckets - 1 else (b + 1) * bucket_size
            bucket = sorted_indices[start:end]
            for i in range(0, len(bucket), batch_size):
                self.batches.append(bucket[i:i + batch_size])

    def __iter__(self):
        for batch in self.batches:
            yield batch

    def __len__(self):
        return len(self.batches)


def collate_fn(batch):
    ids = [s[0] for s in batch]
    texts = [s[1] for s in batch]
    tok = tokenizer(
        texts, max_length=MAX_SOURCE_LENGTH,
        padding=True, truncation=True, return_tensors='pt',
    )
    return ids, tok


dataset = AkkadianDataset(test_df['id'].tolist(), test_df['input_text'].tolist())
sampler = BucketBatchSampler(dataset, BATCH_SIZE, num_buckets=4)
loader = DataLoader(
    dataset,
    batch_sampler=sampler,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=NUM_WORKERS > 0,
)
print(f"DataLoader: {len(loader)} batches (batch_size={BATCH_SIZE})")

In [ ]:
# ============================================================
# INFERENCE - Multi-Model Ensemble + Massive MBR
# ============================================================

def generate_candidates(model, tok, input_ids, attn_mask, device):
    """Generate diverse candidate pool from one model."""
    B = input_ids.shape[0]
    pools = [[] for _ in range(B)]

    gen_common = dict(
        max_new_tokens=MAX_NEW_TOKENS,
        repetition_penalty=REPETITION_PENALTY,
        use_cache=True,
    )

    # 1. Beam search candidates (high quality, low diversity)
    nb = max(NUM_BEAMS, MBR_NUM_BEAM_CANDS)
    beam_out = model.generate(
        input_ids=input_ids, attention_mask=attn_mask,
        do_sample=False, num_beams=nb,
        num_return_sequences=MBR_NUM_BEAM_CANDS,
        length_penalty=LENGTH_PENALTY,
        early_stopping=True,
        **gen_common,
    )
    beam_txt = tok.batch_decode(beam_out, skip_special_tokens=True)
    for i in range(B):
        pools[i].extend(beam_txt[i * MBR_NUM_BEAM_CANDS:(i + 1) * MBR_NUM_BEAM_CANDS])

    # 2. Diverse sampling (varied temperatures and top_p)
    for temp, top_p, count in MBR_SAMPLE_CONFIGS:
        if count <= 0:
            continue
        samp_out = model.generate(
            input_ids=input_ids, attention_mask=attn_mask,
            do_sample=True, num_beams=1,
            temperature=temp, top_p=top_p,
            num_return_sequences=count,
            **gen_common,
        )
        samp_txt = tok.batch_decode(samp_out, skip_special_tokens=True)
        for i in range(B):
            pools[i].extend(samp_txt[i * count:(i + 1) * count])

    return pools


# === Phase 1: Primary model ===
t0 = time.time()
candidate_pools = {}  # id -> list of candidate strings

print(f"{'=' * 60}")
print(f"Phase 1: Primary model - {model_registry[0]['name']}")
print(f"{'=' * 60}")

with torch.inference_mode():
    for batch_idx, (batch_ids, tokenized) in enumerate(tqdm(loader, desc='Primary')):
        try:
            input_ids = tokenized.input_ids.to(device_0)
            attn_mask = tokenized.attention_mask.to(device_0)

            if USE_MBR:
                pools = generate_candidates(
                    primary_model, primary_tokenizer,
                    input_ids, attn_mask, device_0,
                )
                for bid, pool in zip(batch_ids, pools):
                    candidate_pools.setdefault(bid, []).extend(pool)
            else:
                out = primary_model.generate(
                    input_ids=input_ids, attention_mask=attn_mask,
                    num_beams=NUM_BEAMS, length_penalty=LENGTH_PENALTY,
                    max_new_tokens=MAX_NEW_TOKENS, early_stopping=True,
                    use_cache=True,
                )
                texts = primary_tokenizer.batch_decode(out, skip_special_tokens=True)
                for bid, txt in zip(batch_ids, texts):
                    candidate_pools.setdefault(bid, []).append(txt)

            if torch.cuda.is_available() and batch_idx % 10 == 0:
                torch.cuda.empty_cache()

        except Exception as e:
            print(f"  Batch {batch_idx} error: {e}")
            for bid in batch_ids:
                candidate_pools.setdefault(bid, []).append('')

elapsed1 = time.time() - t0
avg_cands = np.mean([len(v) for v in candidate_pools.values()])
print(f"\nPrimary done: {elapsed1:.0f}s, avg {avg_cands:.1f} candidates/sample")

# Checkpoint after primary model
cp_df = pd.DataFrame(
    [(sid, candidate_pools[sid][0] if candidate_pools[sid] else '')
     for sid in sorted(candidate_pools.keys())],
    columns=['id', 'translation'],
)
cp_df.to_csv(Path(OUTPUT_DIR) / 'checkpoint_primary.csv', index=False)
print(f"Checkpoint saved: checkpoint_primary.csv")


# === Phase 2: Additional models (if any) ===
if len(model_registry) > 1:
    use_secondary_gpu = (device_1 != device_0)

    for model_idx in range(1, len(model_registry)):
        mcfg = model_registry[model_idx]
        target_device = device_1 if use_secondary_gpu else device_0

        print(f"\n{'=' * 60}")
        print(f"Phase 2.{model_idx}: {mcfg['name']} (on {target_device})")
        print(f"{'=' * 60}")

        try:
            if not use_secondary_gpu:
                # Free primary to make VRAM space
                del primary_model
                gc.collect()
                torch.cuda.empty_cache()

            sec_tokenizer = AutoTokenizer.from_pretrained(mcfg['path'])
            sec_model = AutoModelForSeq2SeqLM.from_pretrained(
                mcfg['path'], torch_dtype=torch.float32,
            )
            sec_model = sec_model.to(target_device).eval()

            try:
                from optimum.bettertransformer import BetterTransformer
                sec_model = BetterTransformer.transform(sec_model)
            except:
                pass

            n_p = sum(p.numel() for p in sec_model.parameters())
            print(f"  Loaded: {n_p/1e6:.0f}M params on {target_device}")

            for batch_start in tqdm(
                range(0, len(test_df), BATCH_SIZE),
                desc=f'Model {model_idx + 1}',
            ):
                batch_slice = test_df.iloc[batch_start:batch_start + BATCH_SIZE]
                batch_ids = batch_slice['id'].tolist()
                batch_texts = batch_slice['input_text'].tolist()

                tok = sec_tokenizer(
                    batch_texts, max_length=MAX_SOURCE_LENGTH,
                    padding=True, truncation=True, return_tensors='pt',
                )
                input_ids = tok.input_ids.to(target_device)
                attn_mask = tok.attention_mask.to(target_device)

                try:
                    with torch.inference_mode():
                        pools = generate_candidates(
                            sec_model, sec_tokenizer,
                            input_ids, attn_mask, target_device,
                        )
                    for bid, pool in zip(batch_ids, pools):
                        candidate_pools.setdefault(bid, []).extend(pool)
                except Exception as e:
                    print(f"  Batch error: {e}")

            del sec_model, sec_tokenizer
            gc.collect()
            torch.cuda.empty_cache()

        except Exception as e:
            print(f"  FAILED to load {mcfg['name']}: {e}")
            import traceback
            traceback.print_exc()

    # Reload primary if we freed it
    if not use_secondary_gpu:
        print("\nReloading primary model...")
        primary_model = AutoModelForSeq2SeqLM.from_pretrained(
            model_registry[0]['path'], torch_dtype=torch.float32,
        )
        primary_model = primary_model.to(device_0).eval()


# === Phase 3: MBR Selection + Postprocessing ===
print(f"\n{'=' * 60}")
print(f"Phase 3: MBR selection ({len(candidate_pools)} samples)")
print(f"{'=' * 60}")

results = []
for sid in tqdm(sorted(candidate_pools.keys()), desc='MBR'):
    cands = candidate_pools[sid]
    if USE_MBR and len(cands) > 1:
        best = mbr_pick(cands, utility=MBR_UTILITY)
    elif cands:
        best = cands[0]
    else:
        best = ''
    results.append((sid, best))

# Postprocess
raw_translations = [r[1] for r in results]
cleaned = postprocess_batch(raw_translations)
results = [(results[i][0], cleaned[i]) for i in range(len(results))]

elapsed_total = time.time() - t0
avg_cands_final = np.mean([len(v) for v in candidate_pools.values()])
print(f"\nDone: {len(results)} translations in {elapsed_total:.0f}s")
print(f"  ({elapsed_total / max(len(results), 1):.1f}s per sample)")
print(f"  Avg candidates per sample: {avg_cands_final:.1f}")

In [ ]:
# ============================================================
# SAVE SUBMISSION
# ============================================================

sub = pd.DataFrame(results, columns=['id', 'translation'])
sub = sub.sort_values('id').reset_index(drop=True)
sub['translation'] = sub['translation'].fillna('')

# Validation
empty = sub['translation'].str.strip().eq('').sum()
lengths = sub['translation'].str.len()

print(f"\n{'=' * 60}")
print(f"SUBMISSION SUMMARY")
print(f"{'=' * 60}")
print(f"Total translations: {len(sub)}")
print(f"Empty translations: {empty} ({empty / max(1, len(sub)) * 100:.1f}%)")
print(f"Translation lengths:")
print(f"  Mean:   {lengths.mean():.0f} chars")
print(f"  Median: {lengths.median():.0f} chars")
print(f"  Min:    {lengths.min()} chars")
print(f"  Max:    {lengths.max()} chars")

# Quality checks
has_big_gap = sub['translation'].str.contains('<big_gap>', regex=False).sum()
has_h_hook = sub['translation'].str.contains('\u1e2b', regex=False).sum()
print(f"\nQuality checks:")
print(f"  Contains <big_gap>: {has_big_gap} (should be 0)")
print(f"  Contains \u1e2b: {has_h_hook} (should be 0)")

# Show samples
print(f"\nSample translations:")
sample_indices = [0, len(sub) // 4, len(sub) // 2, 3 * len(sub) // 4, len(sub) - 1]
for idx in sample_indices:
    if 0 <= idx < len(sub):
        row = sub.iloc[idx]
        print(f"  ID {int(row['id']):4d}: {str(row['translation'])[:80]}")

# Save
sub_path = Path(OUTPUT_DIR) / 'submission.csv'
sub.to_csv(sub_path, index=False)
print(f"\nSaved: {sub_path} ({len(sub)} rows)")
print(f"Total elapsed: {elapsed_total:.0f}s ({elapsed_total / 60:.1f} min)")